# PySpark coding challenges

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/pyspark/pyspark-coding-challenges.ipynb)

Use these drills to practice the transformations that recur in pipeline code reviews. Every problem has a sample input and expected output. Try the exercise cell before opening the solution, then check edge cases such as duplicate keys, nulls, and ties.

**Start with a workflow.** For cleaning, try problems 8, 9, 4, and 5. For reconciliation, try 2, 15, 20, and 32. For change feeds, try 16, 22, 33, and 26. For time series, try 17, 18, 34, and 36. Problems run from easy to hard. Use the expected outputs to check what one row represents before comparing individual values.

Each technique is taught in [PySpark lessons 2 to 11](pyspark-00-start-here.ipynb) and summarized in the [PySpark reference](pyspark-reference.ipynb).

Run setup once. The examples use local Spark 4.0 and UTC. The handbook site keeps solutions collapsed until you open them.

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0

In [ ]:
import os
import sys
import time

# Workers must use this notebook's interpreter, and every timestamp here is UTC.
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["TZ"] = "UTC"
time.tzset()

from pyspark.sql import SparkSession, Window, functions as F

spark = (
    SparkSession.builder.appName("pyspark-challenges")
    .master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # tiny data needs few partitions
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
MAX_TS = F.lit("9999-12-31 00:00:00").cast("timestamp")
print("Spark", spark.version)

## Work through a challenge

Read the input schema and write down what one output row represents. Pick a small example you can solve by hand, then implement the transformation and compare the result. Use the published checks to test the stated requirements, including ties, nulls, and missing matches where they apply.

**Example.** In a latest-record challenge, two rows for customer 7 share a timestamp. If their source sequences are 10 and 11, the expected output is one row for customer 7 with sequence 11. Check that your result stays the same if you reverse the input order.

## Problem 1 - Tier orders with select, withColumn and when / otherwise

**Difficulty:** Easy · about 10 min

Orders arrive with an integer `amount_cents`, a `status` and a `country`. Build the reporting view.

**Statement.** Return one row per order with `order_id`, `amount_usd`, `tier`, `is_revenue` and `country_group`,
ordered by `order_id`.

Requirements:

- `amount_usd` is `amount_cents / 100`, rounded to two decimals.
- `tier` is `large` when `amount_usd >= 500`, `medium` when it is at least 100, and `small` otherwise.
- `is_revenue` is true only for `status = 'paid'`. Refunded and pending orders are not revenue.
- `country_group` is `unknown` for a null country, `US` for US, and `international` for everything else.

**Sample input**

**`orders`**

| order_id | customer_id | country | status | amount_cents |
| --- | --- | --- | --- | --- |
| 1 | c1 | US | paid | 12500 |
| 2 | c2 | DE | refunded | 4000 |
| 3 | c3 | US | paid | 99900 |
| 4 | c4 | null | pending | 2500 |
| 5 | c5 | IN | paid | 60000 |

**Expected output**

| order_id | amount_usd | tier | is_revenue | country_group |
| --- | --- | --- | --- | --- |
| 1 | 125.0 | medium | true | US |
| 2 | 40.0 | small | false | international |
| 3 | 999.0 | large | true | US |
| 4 | 25.0 | small | false | unknown |
| 5 | 600.0 | large | true | international |

**Reading the result.** Order 1 is paid and worth $125, so it contributes revenue and belongs to the medium tier. A refunded order keeps its amount but is not revenue.

In [ ]:
# Create the sample DataFrames for this problem.
orders = spark.createDataFrame([
    (1, "c1", "US", "paid", 12500),
    (2, "c2", "DE", "refunded", 4000),
    (3, "c3", "US", "paid", 99900),
    (4, "c4", None, "pending", 2500),
    (5, "c5", "IN", "paid", 60000),
], ["order_id", "customer_id", "country", "status", "amount_cents"])

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
result = (
    orders
    .withColumn("amount_usd", F.round(F.col("amount_cents") / 100, 2))
    .withColumn(
        "tier",
        F.when(F.col("amount_usd") >= 500, "large")
        .when(F.col("amount_usd") >= 100, "medium")
        .otherwise("small"),
    )
    .withColumn("is_revenue", F.col("status") == "paid")
    .withColumn(
        "country_group",
        F.when(F.col("country").isNull(), "unknown")
        .when(F.col("country") == "US", "US")
        .otherwise("international"),
    )
    .select("order_id", "amount_usd", "tier", "is_revenue", "country_group")
    .orderBy("order_id")
)

result.show()

## Problem 2 - Semi and anti joins: customers with and without orders

**Difficulty:** Easy · about 10 min

Customers and orders. Some customers have several orders, some none, and one order has no customer.

**Statement.** Return the customers who have at least one order, once each, and the customers who have none.

Requirements:

- Use a left semi join for customers with orders, so each customer appears once even when it has several orders.
- Use a left anti join for customers without orders.
- Show that a plain inner join would repeat a customer, and count those rows.

**Sample input**

**`customers`**

| customer_id | name |
| --- | --- |
| 1 | ana |
| 2 | ben |
| 3 | cy |
| 4 | dee |

**`orders_small`**

| order_id | customer_id |
| --- | --- |
| 10 | 1 |
| 11 | 1 |
| 12 | 3 |
| 13 | null |

**Expected output**

**Customers with orders**

| customer_id | name |
| --- | --- |
| 1 | ana |
| 3 | cy |

**Customers without orders**

| customer_id | name |
| --- | --- |
| 2 | ben |
| 4 | dee |

**Reading the result.** Customer ana has two orders but appears once in the with-orders result. The order with a null customer key matches no customer. A plain inner join would produce three rows.

In [ ]:
# Create the sample DataFrames for this problem.
customers = spark.createDataFrame([(1, "ana"), (2, "ben"), (3, "cy"), (4, "dee")], ["customer_id", "name"])
orders_small = spark.createDataFrame([(10, 1), (11, 1), (12, 3), (13, None)], ["order_id", "customer_id"])

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
with_orders = customers.join(orders_small, "customer_id", "left_semi")

without_orders = customers.join(orders_small, "customer_id", "left_anti")

inner_rows = customers.join(orders_small, "customer_id", "inner").count()

with_orders.orderBy("customer_id").show()

without_orders.orderBy("customer_id").show()

print("Inner join rows:", inner_rows)

## Problem 3 - Self join: employees who out-earn their manager

**Difficulty:** Easy · about 10 min

Employees point at their manager through `manager_id`. The top manager has no manager.

**Statement.** List the employees who earn more than their manager, with both salaries. Then count the direct reports
of each manager.

Requirements:

- Join the table to itself on `manager_id = emp_id`, using two aliases.
- Leave out employees without a manager.
- Count direct reports for every manager that has at least one.

**Sample input**

**`staff`**

| emp_id | name | manager_id | salary |
| --- | --- | --- | --- |
| 1 | Priya | null | 210 |
| 2 | Omar | 1 | 150 |
| 3 | Lena | 1 | 170 |
| 4 | Tom | 2 | 160 |
| 5 | Mei | 3 | 90 |
| 6 | Ravi | 3 | 200 |

**Expected output**

**Employees above their manager**

| name | manager_name | salary | manager_salary |
| --- | --- | --- | --- |
| Ravi | Lena | 200 | 170 |
| Tom | Omar | 160 | 150 |

**Direct reports**

| manager_id | count |
| --- | --- |
| 1 | 2 |
| 2 | 1 |
| 3 | 2 |

**Reading the result.** Ravi earns 200 while manager Lena earns 170. Direct-report counts are a separate result: Lena manages Mei and Ravi, whether or not each earns more than she does.

In [ ]:
# Create the sample DataFrames for this problem.
staff = spark.createDataFrame([
    (1, "Priya", None, 210),
    (2, "Omar", 1, 150),
    (3, "Lena", 1, 170),
    (4, "Tom", 2, 160),
    (5, "Mei", 3, 90),
    (6, "Ravi", 3, 200),
], ["emp_id", "name", "manager_id", "salary"])

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
e = staff.alias("e")

m = staff.alias("m")

over_manager = (
    e.join(m, F.col("e.manager_id") == F.col("m.emp_id"), "inner")
    .filter(F.col("e.salary") > F.col("m.salary"))
    .select(
        F.col("e.name").alias("name"),
        F.col("m.name").alias("manager_name"),
        F.col("e.salary").alias("salary"),
        F.col("m.salary").alias("manager_salary"),
    )
    .orderBy("name")
)

over_manager.show()

reports = staff.filter(F.col("manager_id").isNotNull()).groupBy("manager_id").count()

reports.orderBy("manager_id").show()

## Problem 4 - Clean and transform customer records

**Difficulty:** Easy · about 10 min

Trim and lowercase email addresses, replace a null or blank email with `unknown`, and assign `adult` for age 18 or greater, `minor` below 18, and `unknown` for a null age. Return `id`, `email_clean`, and `age_group`, one row per customer.

**Sample input**

**`customers`**

| id | email | age |
| --- | --- | --- |
| 1 | `"  AVA@EXAMPLE.COM "` | 22 |
| 2 | `"  "` | 16 |
| 3 | null | null |

**Expected output**

| id | email_clean | age_group |
| --- | --- | --- |
| 1 | ava@example.com | adult |
| 2 | unknown | minor |
| 3 | unknown | unknown |

**Reading the result.** Blank and null emails both become unknown, but a missing age remains its own unknown group rather than becoming minor.

In [ ]:
# Create the sample DataFrames for this problem.
customers = spark.createDataFrame([(1, '  AVA@EXAMPLE.COM ', 22), (2, '  ', 16), (3, None, None)], 'id int, email string, age int')

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
result = customers.select(
    'id',
    F.when(F.trim(F.col('email')) == '', 'unknown')
     .otherwise(F.coalesce(F.lower(F.trim('email')), F.lit('unknown'))).alias('email_clean'),
    F.when(F.col('age').isNull(), 'unknown').when(F.col('age') >= 18, 'adult')
     .otherwise('minor').alias('age_group'))

result.orderBy('id').show(truncate=False)

## Problem 5 - Parse dates and calculate elapsed days

**Difficulty:** Easy · about 10 min

Parse `placed` and `shipped` as dates, then calculate `days_to_ship`. Preserve null shipment dates and return `order_id`, `placed_date`, and `days_to_ship`.

**Sample input**

**`shipments`**

| order_id | placed | shipped |
| --- | --- | --- |
| o1 | 2026-01-30 | 2026-02-02 |
| o2 | 2026-02-28 | null |
| o3 | 2026-03-01 | 2026-03-01 |

**Expected output**

| order_id | placed_date | days_to_ship |
| --- | --- | --- |
| o1 | 2026-01-30 | 3 |
| o2 | 2026-02-28 | null |
| o3 | 2026-03-01 | 0 |

**Reading the result.** The first shipment crosses a month boundary but still takes three calendar days. An unshipped order has a null elapsed time, not zero.

In [ ]:
# Create the sample DataFrames for this problem.
shipments = spark.createDataFrame([('o1', '2026-01-30', '2026-02-02'), ('o2', '2026-02-28', None), ('o3', '2026-03-01', '2026-03-01')], ['order_id', 'placed', 'shipped'])

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
result = shipments.select('order_id', F.to_date('placed').alias('placed_date'), F.datediff(F.to_date('shipped'), F.to_date('placed')).alias('days_to_ship'))

result.orderBy('order_id').show()

## Problem 6 - Left joins and missing matches

**Difficulty:** Easy · about 10 min

List every customer with the number of orders. Return zero for customers with no orders, and do not count the null placeholder produced by a left join.

**Sample input**

**`join_customers`**

| customer_id | name |
| --- | --- |
| 1 | Ava |
| 2 | Bo |
| 3 | Cy |

**`join_orders`**

| order_id | customer_id |
| --- | --- |
| 101 | 1 |
| 102 | 1 |
| 103 | 2 |

**Expected output**

| customer_id | name | order_count |
| --- | --- | --- |
| 1 | Ava | 2 |
| 2 | Bo | 1 |
| 3 | Cy | 0 |

**Reading the result.** Cy has no matching order and must still appear with count 0. Count real order IDs rather than the null placeholder row produced by the left join.

In [ ]:
# Create the sample DataFrames for this problem.
join_customers = spark.createDataFrame([(1, 'Ava'), (2, 'Bo'), (3, 'Cy')], ['customer_id', 'name'])
join_orders = spark.createDataFrame([(101, 1), (102, 1), (103, 2)], ['order_id', 'customer_id'])

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
result = (join_customers.join(join_orders, 'customer_id', 'left')
          .groupBy('customer_id', 'name').agg(F.count('order_id').alias('order_count')))

result.orderBy('customer_id').show()

## Problem 7 - Anti join: customers without purchases

**Difficulty:** Easy · about 10 min

Return the customer IDs with no matching order using a left anti join. A null key in the order table must not exclude all customers.

**Sample input**

**`anti_customers`**

| customer_id |
| --- |
| 1 |
| 2 |
| 3 |

**`anti_orders`**

| customer_id |
| --- |
| 1 |
| null |

**Expected output**

| customer_id |
| --- |
| 2 |
| 3 |

**Reading the result.** Only customer 1 has a matching purchase. The null order key does not remove customers 2 and 3 from an anti join.

In [ ]:
# Create the sample DataFrames for this problem.
anti_customers = spark.createDataFrame([(1,), (2,), (3,)], 'customer_id int')
anti_orders = spark.createDataFrame([(1,), (None,)], 'customer_id int')

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
result = anti_customers.join(anti_orders, 'customer_id', 'left_anti')

result.orderBy('customer_id').show()

## Problem 8 - Null handling: coalesce, dropna, null-safe equality and null counts

**Difficulty:** Medium · about 15 min

A profile snapshot has missing values, and some rows have no user id at all.

**Statement.** Clean the snapshot and report how many values are missing in each column.

Requirements:

- Drop rows whose `user_id` is null.
- `display_name` is the first non-null of `nickname`, `first_name`, then the text `anonymous`.
- `score_filled` is `score`, with a null replaced by `0.0`.
- `is_pro` is true when `plan` is `pro`. A null plan must give `false`, not null, so use a null-safe comparison.
- Return null counts for the original columns after dropping missing user IDs, before filling or projecting values.

**Sample input**

**`profiles`**

| user_id | first_name | nickname | score | plan |
| --- | --- | --- | --- | --- |
| 1 | Ana | ani | 88.5 | pro |
| 2 | Ben | null | null | null |
| 3 | null | null | 42.0 | free |
| null | Cy | cy | 10.0 | pro |
| 5 | null | dee | null | pro |

**Expected output**

**Cleaned profiles**

| user_id | display_name | score_filled | is_pro |
| --- | --- | --- | --- |
| 1 | ani | 88.5 | true |
| 2 | Ben | 0.0 | false |
| 3 | anonymous | 42.0 | false |
| 5 | dee | 0.0 | true |

**Null counts**

| first_name | nickname | plan | score | user_id |
| --- | --- | --- | --- | --- |
| 2 | 2 | 1 | 2 | 0 |

**Reading the result.** The row without a user ID is removed first. User 3 has neither name, so the display name becomes anonymous; an unknown plan is false rather than null. Count nulls in the retained original columns, before filling or projecting them.

In [ ]:
# Create the sample DataFrames for this problem.
profiles = spark.createDataFrame([
    (1, "Ana", "ani", 88.5, "pro"),
    (2, "Ben", None, None, None),
    (3, None, None, 42.0, "free"),
    (None, "Cy", "cy", 10.0, "pro"),
    (5, None, "dee", None, "pro"),
], ["user_id", "first_name", "nickname", "score", "plan"])

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
cleaned = profiles.dropna(subset=["user_id"])

result = (
    cleaned
    .withColumn("display_name", F.coalesce("nickname", "first_name", F.lit("anonymous")))
    .withColumn("score_filled", F.coalesce("score", F.lit(0.0)))
    .withColumn("is_pro", F.col("plan").eqNullSafe("pro"))
    .select("user_id", "display_name", "score_filled", "is_pro")
    .orderBy("user_id")
)

result.show()

null_counts = cleaned.agg(
    *[F.sum(F.col(column).isNull().cast("int")).alias(column) for column in cleaned.columns]
).first().asDict()

print("Null counts before filling:", null_counts)

## Problem 9 - String functions: trim, regex extraction, split and padding

**Difficulty:** Medium · about 15 min

Raw addresses and locations arrive as free text, with stray spaces, mixed case and optional parts.

**Statement.** Clean the emails and split the locations into city, country and postal code.

Requirements:

- `email_clean` is the email trimmed and lower-cased. A null stays null.
- `email_domain` is the text after the last `@`.
- `email_valid` is true when `email_clean` matches a simple pattern: a local part, one `@`, a domain, and a top-level
  domain of two or more lower-case letters. A null stays null.
- Split `location_raw` on `|` with optional spaces. The city is part 1, the country part 2, the postal code part 3.
  A missing part is null, not an error.
- `postal_padded` is the postal code left-padded with zeros to six characters.
- `slug` is `city-country` in lower case, for example `berlin-germany`.

**Sample input**

**`raw_addresses`**

| id | email_raw | location_raw |
| --- | --- | --- |
| 1 | `"  alice.SMITH@Example.COM  "` | Berlin &#124; Germany &#124; 10115 |
| 2 | bob@example.org | Paris &#124; France &#124; 75001 |
| 3 | null | Lima &#124; Peru |
| 4 | carol@@example.net | Oslo &#124; Norway &#124; 0150 |

**Expected output**

| id | email_clean | email_domain | email_valid | city | country | postal_padded | slug |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 1 | alice.smith@example.com | example.com | true | Berlin | Germany | 010115 | berlin-germany |
| 2 | bob@example.org | example.org | true | Paris | France | 075001 | paris-france |
| 3 | null | null | null | Lima | Peru | null | lima-peru |
| 4 | carol@@example.net | example.net | false | Oslo | Norway | 000150 | oslo-norway |

**Reading the result.** The missing postal component for Lima stays null. Extracting a domain is separate from validating the whole address: carol@@example.net has an extractable domain but is invalid.

In [ ]:
# Create the sample DataFrames for this problem.
raw_addresses = spark.createDataFrame([
    (1, "  alice.SMITH@Example.COM  ", "Berlin | Germany | 10115"),
    (2, "bob@example.org", "Paris | France | 75001"),
    (3, None, "Lima | Peru"),
    (4, "carol@@example.net", "Oslo | Norway | 0150"),
], ["id", "email_raw", "location_raw"])

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
EMAIL_PATTERN = r"^[^@\s]+@[^@\s]+\.[a-z]{2,}$"

parts = F.split("location_raw", r"\s*\|\s*")

result = (
    raw_addresses
    .withColumn("email_clean", F.lower(F.trim("email_raw")))
    .withColumn("email_domain", F.regexp_extract("email_clean", r"@([^@]+)$", 1))
    .withColumn("email_valid", F.col("email_clean").rlike(EMAIL_PATTERN))
    .withColumn("city", F.try_element_at(parts, F.lit(1)))
    .withColumn("country", F.try_element_at(parts, F.lit(2)))
    .withColumn("postal_code", F.try_element_at(parts, F.lit(3)))
    .withColumn("postal_padded", F.lpad("postal_code", 6, "0"))
    .withColumn("slug", F.lower(F.concat_ws("-", "city", "country")))
    .select("id", "email_clean", "email_domain", "email_valid", "city", "country", "postal_padded", "slug")
    .orderBy("id")
)

result.show(truncate=False)

## Problem 10 - Date functions: renewals, month ends and weekdays

**Difficulty:** Medium · about 15 min

Subscriptions start on a date and may be cancelled. Every report uses one fixed as-of date, `2026-03-31`.

**Statement.** Derive the renewal date, the month end and the length of each subscription.

Requirements:

- `renewal_date` is one calendar month after `started_on`. Spark clamps the day, so 31 January gives 28 February.
- `month_end` is the last day of the month in which the subscription started.
- `days_active` is the number of days from `started_on` to `cancelled_on`, or to the as-of date when it is still open.
- `is_active` is true when `cancelled_on` is null.
- `started_on_weekend` is true when the subscription started on a Saturday or Sunday.

**Sample input**

**`subscriptions`**

| sub_id | started_on | cancelled_on |
| --- | --- | --- |
| s1 | 2026-01-31 | 2026-03-15 |
| s2 | 2026-02-10 | null |
| s3 | 2026-03-07 | 2026-03-08 |
| s4 | 2026-03-14 | 2026-03-14 |

**Expected output**

| sub_id | renewal_date | month_end | days_active | is_active | started_on_weekend |
| --- | --- | --- | --- | --- | --- |
| s1 | 2026-02-28 | 2026-01-31 | 43 | false | true |
| s2 | 2026-03-10 | 2026-02-28 | 49 | true | false |
| s3 | 2026-04-07 | 2026-03-31 | 1 | false | true |
| s4 | 2026-04-14 | 2026-03-31 | 0 | false | true |

**Reading the result.** Renewal adds a calendar month, not 30 days: January 31 clamps to February 28. An open subscription uses the fixed report date, keeping the example reproducible.

In [ ]:
# Create the sample DataFrames for this problem.
subscriptions = spark.createDataFrame([
    ("s1", "2026-01-31", "2026-03-15"),
    ("s2", "2026-02-10", None),
    ("s3", "2026-03-07", "2026-03-08"),
    ("s4", "2026-03-14", "2026-03-14"),
], ["sub_id", "started_on", "cancelled_on"])

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
AS_OF = F.to_date(F.lit("2026-03-31"))

subs = (
    subscriptions
    .withColumn("started_on", F.to_date("started_on"))
    .withColumn("cancelled_on", F.to_date("cancelled_on"))
)

result = (
    subs
    .withColumn("renewal_date", F.add_months("started_on", 1))
    .withColumn("month_end", F.last_day("started_on"))
    .withColumn("days_active", F.datediff(F.coalesce("cancelled_on", AS_OF), "started_on"))
    .withColumn("is_active", F.col("cancelled_on").isNull())
    .withColumn("started_on_weekend", F.dayofweek("started_on").isin(1, 7))  # 1 = Sunday, 7 = Saturday
    .select("sub_id", "renewal_date", "month_end", "days_active", "is_active", "started_on_weekend")
    .orderBy("sub_id")
)

result.show()

## Problem 11 - explode, posexplode_outer and keeping empty arrays

**Difficulty:** Medium · about 15 min

Articles carry an array of tags. Some have no tags, and one has a null array.

**Statement.** Produce one row per article and tag, with the tag's position in the array. Articles without tags must
still appear, once, with null position and tag.

Requirements:

- Positions start at 0.
- Use an outer generator so that empty and null arrays keep their article.
- Count how many articles carry each tag. The null tag is counted for the articles without tags.

**Sample input**

**`articles`**

| article_id | title | tags |
| --- | --- | --- |
| 1 | spark tips | `["spark", "sql"]` |
| 2 | lakehouse | `["delta", "iceberg", "spark"]` |
| 3 | draft | `[]` |
| 4 | untagged | null |

**Expected output**

**Article tags**

| article_id | tag_pos | tag |
| --- | --- | --- |
| 1 | 0 | spark |
| 1 | 1 | sql |
| 2 | 0 | delta |
| 2 | 1 | iceberg |
| 2 | 2 | spark |
| 3 | null | null |
| 4 | null | null |

**Tag counts**

| tag | count |
| --- | --- |
| null | 2 |
| delta | 1 |
| iceberg | 1 |
| spark | 2 |
| sql | 1 |

**Reading the result.** The empty and null arrays each preserve their article with a null position and tag. The supplied tag arrays contain no duplicate tag within one article.

In [ ]:
# Create the sample DataFrames for this problem.
articles = spark.createDataFrame([
    (1, "spark tips", ["spark", "sql"]),
    (2, "lakehouse", ["delta", "iceberg", "spark"]),
    (3, "draft", []),
    (4, "untagged", None),
], ["article_id", "title", "tags"])

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
exploded = articles.select("article_id", F.posexplode_outer("tags").alias("tag_pos", "tag"))

exploded.orderBy("article_id", "tag_pos").show()

tag_counts = {r["tag"]: r["count"] for r in exploded.groupBy("tag").count().collect()}

print("Tag counts:", tag_counts)

## Problem 12 - Pivot to wide, then unpivot back with stack

**Difficulty:** Medium · about 15 min

Quarterly sales arrive in long form: one row per region, quarter and sale.

**Statement.** Build a wide report with one column per quarter, then turn the wide report back into long form.

Requirements:

- Pivot with the four quarters given explicitly, so the column order is fixed and no extra Spark job is needed.
- Missing region and quarter combinations must show `0`, not null.
- Unpivot the wide report with `stack` and drop the zero cells. The total must match the original sales.

**Sample input**

**`sales`**

| region | quarter | amount |
| --- | --- | --- |
| north | Q1 | 100 |
| north | Q2 | 150 |
| north | Q2 | 50 |
| south | Q1 | 80 |
| south | Q3 | 120 |
| west | Q1 | 30 |
| west | Q2 | 40 |
| west | Q3 | 60 |
| west | Q4 | 90 |

**Expected output**

**Wide sales**

| region | Q1 | Q2 | Q3 | Q4 |
| --- | --- | --- | --- | --- |
| north | 100 | 200 | 0 | 0 |
| south | 80 | 0 | 120 | 0 |
| west | 30 | 40 | 60 | 90 |

**Long sales**

| region | quarter | amount |
| --- | --- | --- |
| north | Q1 | 100 |
| north | Q2 | 200 |
| south | Q1 | 80 |
| south | Q3 | 120 |
| west | Q1 | 30 |
| west | Q2 | 40 |
| west | Q3 | 60 |
| west | Q4 | 90 |

**Reading the result.** North has two Q2 sales, so its wide Q2 cell is 200. Unpivoting returns aggregated region/quarter cells, not the original individual sales; both reports total 720.

In [ ]:
# Create the sample DataFrames for this problem.
sales = spark.createDataFrame([
    ("north", "Q1", 100), ("north", "Q2", 150), ("north", "Q2", 50),
    ("south", "Q1", 80), ("south", "Q3", 120),
    ("west", "Q1", 30), ("west", "Q2", 40), ("west", "Q3", 60), ("west", "Q4", 90),
], ["region", "quarter", "amount"])

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
wide = (
    sales.groupBy("region")
    .pivot("quarter", ["Q1", "Q2", "Q3", "Q4"])
    .agg(F.sum("amount"))
    .fillna(0)
    .orderBy("region")
)

wide.show()

long = (
    wide.selectExpr("region", "stack(4, 'Q1', Q1, 'Q2', Q2, 'Q3', Q3, 'Q4', Q4) AS (quarter, amount)")
    .filter("amount <> 0")
)

long.orderBy("region", "quarter").show()

## Problem 13 - Replace a Python UDF with built-in functions

**Difficulty:** Medium · about 15 min

A teammate normalised phone numbers with a Python UDF, which calls Python once per row. Rewrite the logic with
built-in functions.

**Statement.** Keep the digits of `raw`. Ten digits become `+1` followed by the digits. Eleven digits that start with
`1` become `+` followed by the digits. Anything else is null. The output must equal the UDF's output.

Requirements:

- Do not use `udf` or `pandas_udf` in your solution.
- Show that the built-in plan has no Python step (`BatchEvalPython`).
- Explain when a UDF is still the right tool, for example for logic that has no built-in equivalent.

**Sample input**

**`phones`**

| id | raw |
| --- | --- |
| a | (415) 555-0100 |
| b | 1-415-555-0100 |
| c | +44 20 7946 0958 |
| d | 555-0100 |
| e | null |

**Expected output**

| id | raw | phone_e164 |
| --- | --- | --- |
| a | (415) 555-0100 | +14155550100 |
| b | 1-415-555-0100 | +14155550100 |
| c | +44 20 7946 0958 | null |
| d | 555-0100 | null |
| e | null | null |

**Reading the result.** The two US formats normalize to the same number. The UK example and short number are unsupported, so they become null; this is a deliberately limited normalization rule, not general phone validation.

In [ ]:
# Create the sample DataFrames for this problem.
import io
from contextlib import redirect_stdout

phones = spark.createDataFrame([
    ("a", "(415) 555-0100"),
    ("b", "1-415-555-0100"),
    ("c", "+44 20 7946 0958"),
    ("d", "555-0100"),
    ("e", None),
], ["id", "raw"])


@F.udf("string")
def legacy_normalize(raw):
    """The teammate's version: one Python call per row."""
    if raw is None:
        return None
    digits = "".join(ch for ch in raw if ch.isdigit())
    if len(digits) == 10:
        return "+1" + digits
    if len(digits) == 11 and digits.startswith("1"):
        return "+" + digits
    return None


legacy = phones.withColumn("phone_e164", legacy_normalize("raw")).orderBy("id")

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
digits = F.regexp_replace("raw", r"\D", "")

builtin = (
    phones.withColumn("digits", digits)
    .withColumn(
        "phone_e164",
        F.when(F.length("digits") == 10, F.concat(F.lit("+1"), F.col("digits")))
        .when((F.length("digits") == 11) & F.col("digits").startswith("1"), F.concat(F.lit("+"), F.col("digits"))),
    )
    .select("id", "raw", "phone_e164")
    .orderBy("id")
)

builtin.show(truncate=False)

plan = io.StringIO()

with redirect_stdout(plan):
    builtin.explain()

builtin.explain()

## Problem 14 - Broadcast joins and proving the plan

**Difficulty:** Medium · about 15 min

A click log has a thousand rows and points at five countries. The country table is tiny.

**Statement.** Count the clicks per country, and show from the physical plan that the join broadcasts the small table.

Requirements:

- Hint the broadcast of the country table with `F.broadcast`.
- Prove it with the plan: the physical plan must contain `BroadcastHashJoin`.
- Explain when broadcasting is wrong, for example when the small side is too large for executor or driver memory.

**Sample input**

**`clicks`**

| country_id | click_id |
| --- | --- |
| 1 | 0 |
| 2 | 1 |
| 3 | 2 |
| 4 | 3 |
| 5 | 4 |

The table above shows the first five generated rows. The full input has click IDs 0–999, with `country_id = click_id % 5 + 1`.

**`countries`**

| country_id | country |
| --- | --- |
| 1 | US |
| 2 | DE |
| 3 | IN |
| 4 | BR |
| 5 | JP |

**Expected output**

| country | clicks |
| --- | --- |
| BR | 200 |
| DE | 200 |
| IN | 200 |
| JP | 200 |
| US | 200 |

**Reading the result.** The generated click IDs are 0 through 999. Their country IDs cycle from 1 through 5, so each country receives 200 clicks. The tiny example demonstrates a plan choice, not a performance benchmark.

In [ ]:
# Create the sample DataFrames for this problem.
clicks = spark.range(0, 1000).select(
    (F.col("id") % 5 + 1).cast("int").alias("country_id"),
    F.col("id").alias("click_id"),
)
countries = spark.createDataFrame([(1, "US"), (2, "DE"), (3, "IN"), (4, "BR"), (5, "JP")], ["country_id", "country"])

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
import io

from contextlib import redirect_stdout

joined = clicks.join(F.broadcast(countries), "country_id")

per_country = joined.groupBy("country").agg(F.count("*").alias("clicks")).orderBy("country")

per_country.show()

plan = io.StringIO()

with redirect_stdout(plan):
    joined.explain()

joined.explain()

## Problem 15 - Full outer reconciliation between a source and a warehouse

**Difficulty:** Medium · about 15 min

A source system and a warehouse should hold the same accounts. They can disagree in three ways: a row is missing on one
side, the balance differs, or the status differs.

**Statement.** Classify every account id that appears on either side.

Requirements:

- Join on `account_id` with a full outer join, so nothing is lost.
- `missing_in_warehouse` when only the source has the account, and `missing_in_source` when only the warehouse does.
- `mismatch` when both sides have it but the balance or status differs. A null status on both sides is equal.
- Otherwise `match`.

**Sample input**

**`source_rows`**

| account_id | balance | status |
| --- | --- | --- |
| A1 | 100.0 | open |
| A2 | 50.0 | closed |
| A3 | 75.0 | null |
| A4 | 20.0 | open |

**`warehouse_rows`**

| account_id | balance | status |
| --- | --- | --- |
| A1 | 100.0 | open |
| A2 | 55.0 | closed |
| A3 | 75.0 | null |
| A5 | 10.0 | open |

**Expected output**

| account_id | check |
| --- | --- |
| A1 | match |
| A2 | mismatch |
| A3 | match |
| A4 | missing_in_warehouse |
| A5 | missing_in_source |

**Reading the result.** A3 matches because its balances agree and both statuses are null. A2 is a value mismatch, while A4 and A5 exist on only one side.

In [ ]:
# Create the sample DataFrames for this problem.
source_rows = spark.createDataFrame([
    ("A1", 100.0, "open"),
    ("A2", 50.0, "closed"),
    ("A3", 75.0, None),
    ("A4", 20.0, "open"),
], ["account_id", "balance", "status"])
warehouse_rows = spark.createDataFrame([
    ("A1", 100.0, "open"),
    ("A2", 55.0, "closed"),
    ("A3", 75.0, None),
    ("A5", 10.0, "open"),
], ["account_id", "balance", "status"])

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
joined = source_rows.alias("s").join(warehouse_rows.alias("w"), "account_id", "full_outer")

recon = (
    joined.select(
        "account_id",
        F.when(F.col("w.balance").isNull(), "missing_in_warehouse")
        .when(F.col("s.balance").isNull(), "missing_in_source")
        .when(
            F.col("s.balance").eqNullSafe(F.col("w.balance")) & F.col("s.status").eqNullSafe(F.col("w.status")),
            "match",
        )
        .otherwise("mismatch")
        .alias("check"),
    )
    .orderBy("account_id")
)

recon.show()

## Problem 16 - row_number, rank and dense_rank with ties

**Difficulty:** Medium · about 15 min

Students have scores in several subjects, and some scores are equal.

**Statement.** Rank the students in each subject three ways, and explain the difference.

Requirements:

- `row_number` gives every row a distinct number. Break score ties by student name, ascending.
- `rank` leaves gaps after a tie. `dense_rank` does not.
- The rank functions must not use the name tie-break, or ties would disappear. Use a separate window for them.

**Sample input**

**`scores`**

| student | subject | score |
| --- | --- | --- |
| ana | math | 95 |
| ben | math | 88 |
| cy | math | 95 |
| dee | math | 80 |
| eli | math | 88 |
| ana | cs | 70 |
| ben | cs | 90 |

**Expected output**

| subject | student | score | row_number | rank | dense_rank |
| --- | --- | --- | --- | --- | --- |
| cs | ben | 90 | 1 | 1 | 1 |
| cs | ana | 70 | 2 | 2 | 2 |
| math | ana | 95 | 1 | 1 | 1 |
| math | cy | 95 | 2 | 1 | 1 |
| math | ben | 88 | 3 | 3 | 2 |
| math | eli | 88 | 4 | 3 | 2 |
| math | dee | 80 | 5 | 5 | 3 |

**Reading the result.** The two math scores of 95 share rank 1. The next score receives rank 3 and dense rank 2, while row numbers remain unique.

In [ ]:
# Create the sample DataFrames for this problem.
scores = spark.createDataFrame([
    ("ana", "math", 95), ("ben", "math", 88), ("cy", "math", 95), ("dee", "math", 80), ("eli", "math", 88),
    ("ana", "cs", 70), ("ben", "cs", 90),
], ["student", "subject", "score"])

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
by_student = Window.partitionBy("subject").orderBy(F.col("score").desc(), "student")

by_score = Window.partitionBy("subject").orderBy(F.col("score").desc())

ranked = (
    scores.select(
        "subject", "student", "score",
        F.row_number().over(by_student).alias("row_number"),
        F.rank().over(by_score).alias("rank"),
        F.dense_rank().over(by_score).alias("dense_rank"),
    )
    .orderBy("subject", "row_number")
)

ranked.show()

## Problem 17 - Days between orders with lag and lead

**Difficulty:** Medium · about 15 min

Customers place orders on several dates. A gap of more than 30 days since the previous order is a churn risk.

**Statement.** For every order, find the previous and the next order date of the same customer, the days since the
previous order, and whether the order is a churn risk.

Requirements:

- Order each customer's rows by `order_date`.
- `days_since_previous` is null for the first order.
- `churn_risk` is true when `days_since_previous` is greater than 30, and false otherwise, including for the first order.

**Sample input**

**`purchases`**

| customer_id | order_date | amount |
| --- | --- | --- |
| c1 | 2026-01-01 | 20.0 |
| c1 | 2026-01-15 | 35.0 |
| c1 | 2026-03-01 | 10.0 |
| c2 | 2026-02-10 | 99.0 |
| c3 | 2026-01-03 | 15.0 |
| c3 | 2026-01-05 | 25.0 |

**Expected output**

| customer_id | order_date | amount | prev_order_date | next_order_date | days_since_previous | is_first_order | churn_risk |
| --- | --- | --- | --- | --- | --- | --- | --- |
| c1 | 2026-01-01 | 20.0 | null | 2026-01-15 | null | true | false |
| c1 | 2026-01-15 | 35.0 | 2026-01-01 | 2026-03-01 | 14 | false | false |
| c1 | 2026-03-01 | 10.0 | 2026-01-15 | null | 45 | false | true |
| c2 | 2026-02-10 | 99.0 | null | null | null | true | false |
| c3 | 2026-01-03 | 15.0 | null | 2026-01-05 | null | true | false |
| c3 | 2026-01-05 | 25.0 | 2026-01-03 | null | 2 | false | false |

**Reading the result.** Customer c1 has a 45-day gap before March 1, so only that row is flagged. A single-order customer has neither a previous nor a next order, and is not flagged.

In [ ]:
# Create the sample DataFrames for this problem.
purchases = spark.createDataFrame([
    ("c1", "2026-01-01", 20.0),
    ("c1", "2026-01-15", 35.0),
    ("c1", "2026-03-01", 10.0),
    ("c2", "2026-02-10", 99.0),
    ("c3", "2026-01-03", 15.0),
    ("c3", "2026-01-05", 25.0),
], ["customer_id", "order_date", "amount"])

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
w = Window.partitionBy("customer_id").orderBy("order_date")

result = (
    purchases
    .withColumn("order_date", F.to_date("order_date"))
    .withColumn("prev_order_date", F.lag("order_date").over(w))
    .withColumn("next_order_date", F.lead("order_date").over(w))
    .withColumn("days_since_previous", F.datediff("order_date", "prev_order_date"))
    .withColumn("is_first_order", F.col("prev_order_date").isNull())
    .withColumn("churn_risk", F.coalesce(F.col("days_since_previous") > 30, F.lit(False)))
    .select(
        "customer_id", "order_date", "amount", "prev_order_date", "next_order_date",
        "days_since_previous", "is_first_order", "churn_risk",
    )
    .orderBy("customer_id", "order_date")
)

result.show()

## Problem 18 - Running totals and moving averages with row and range frames

**Difficulty:** Medium · about 15 min

Daily revenue per store. One store skipped a day.

**Statement.** Add a running total, a three-row moving average, and the revenue of the last three calendar days.

Requirements:

- `running_total` is the sum from the first day to the current row.
- `moving_avg_3` averages the current row and the two rows before it, rounded to two decimals. It counts rows, so a
  missing day is not filled in.
- `sum_last_3_days` sums the revenue of the current calendar day and the two days before it. It uses a range frame on
  the day number, so a missing day contributes nothing to the sum; no row is created for it.

**Sample input**

**`daily`**

| store_id | sale_date | revenue |
| --- | --- | --- |
| s1 | 2026-03-01 | 100.0 |
| s1 | 2026-03-02 | 40.0 |
| s1 | 2026-03-03 | 60.0 |
| s1 | 2026-03-05 | 80.0 |
| s2 | 2026-03-01 | 10.0 |
| s2 | 2026-03-02 | 30.0 |

**Expected output**

| store_id | sale_date | revenue | running_total | moving_avg_3 | sum_last_3_days |
| --- | --- | --- | --- | --- | --- |
| s1 | 2026-03-01 | 100.0 | 100.0 | 100.0 | 100.0 |
| s1 | 2026-03-02 | 40.0 | 140.0 | 70.0 | 140.0 |
| s1 | 2026-03-03 | 60.0 | 200.0 | 66.67 | 200.0 |
| s1 | 2026-03-05 | 80.0 | 280.0 | 60.0 | 140.0 |
| s2 | 2026-03-01 | 10.0 | 10.0 | 10.0 | 10.0 |
| s2 | 2026-03-02 | 30.0 | 40.0 | 20.0 | 40.0 |

**Reading the result.** On March 5, the last three observed rows total 180, giving an average of 60. The three-calendar-day range contains March 3 through 5 and sums to 140.

In [ ]:
# Create the sample DataFrames for this problem.
daily = spark.createDataFrame([
    ("s1", "2026-03-01", 100.0),
    ("s1", "2026-03-02", 40.0),
    ("s1", "2026-03-03", 60.0),
    ("s1", "2026-03-05", 80.0),
    ("s2", "2026-03-01", 10.0),
    ("s2", "2026-03-02", 30.0),
], ["store_id", "sale_date", "revenue"])

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
by_date = Window.partitionBy("store_id").orderBy("sale_date")

last_three_rows = by_date.rowsBetween(-2, 0)

by_day_number = (
    Window.partitionBy("store_id")
    .orderBy("day_number")
    .rangeBetween(-2, 0)
)

result = (
    daily
    .withColumn("sale_date", F.to_date("sale_date"))
    .withColumn("day_number", F.datediff("sale_date", F.to_date(F.lit("1970-01-01"))))
    .withColumn("running_total", F.sum("revenue").over(by_date.rowsBetween(Window.unboundedPreceding, Window.currentRow)))
    .withColumn("moving_avg_3", F.round(F.avg("revenue").over(last_three_rows), 2))
    .withColumn("sum_last_3_days", F.sum("revenue").over(by_day_number))
    .select("store_id", "sale_date", "revenue", "running_total", "moving_avg_3", "sum_last_3_days")
    .orderBy("store_id", "sale_date")
)

result.show()

## Problem 19 - Top-N per group, keeping ties

**Difficulty:** Medium · about 15 min

A product catalogue lists revenue per product. Two products tie at the top of one category, and two tie at the cut-off
of another.

**Statement.** Return the top two revenue levels of every category. Every product at those levels is kept, so a tie
at the cut-off is not dropped.

Requirements:

- Rank within each category by revenue, descending, with `dense_rank`.
- Keep rows whose rank is at most 2.
- Explain why `row_number` would return a different number of rows here.

**Sample input**

**`sales_items`**

| category | product | revenue |
| --- | --- | --- |
| books | Dune | 120.0 |
| books | Emma | 300.0 |
| books | Ulysses | 300.0 |
| books | Odyssey | 40.0 |
| games | Chess | 75.0 |
| games | Go | 20.0 |
| games | Risk | 75.0 |
| toys | Yo-yo | 9.0 |

**Expected output**

| category | product | revenue |
| --- | --- | --- |
| books | Emma | 300.0 |
| books | Ulysses | 300.0 |
| books | Dune | 120.0 |
| games | Chess | 75.0 |
| games | Risk | 75.0 |
| games | Go | 20.0 |
| toys | Yo-yo | 9.0 |

**Reading the result.** Books keeps both products at revenue 300 and Dune at 120. Top two distinct revenue levels can therefore retain more than two products.

In [ ]:
# Create the sample DataFrames for this problem.
sales_items = spark.createDataFrame([
    ("books", "Dune", 120.0), ("books", "Emma", 300.0), ("books", "Ulysses", 300.0), ("books", "Odyssey", 40.0),
    ("games", "Chess", 75.0), ("games", "Go", 20.0), ("games", "Risk", 75.0),
    ("toys", "Yo-yo", 9.0),
], ["category", "product", "revenue"])

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
by_revenue = Window.partitionBy("category").orderBy(F.col("revenue").desc())

top_two_levels = (
    sales_items.withColumn("level", F.dense_rank().over(by_revenue))
    .filter(F.col("level") <= 2)
    .select("category", "product", "revenue")
    .orderBy("category", F.col("revenue").desc(), "product")
)

top_two_levels.show()

## Problem 20 - Aggregations with nulls: count, sum, avg and distinct counts

**Difficulty:** Medium · about 15 min

Payments per merchant. Amounts and methods can be null, and some merchants have no successful payments.

**Statement.** Summarise the payments of each merchant with several aggregations at once.

Requirements:

- `n_rows` counts every payment, and `n_amount` counts only the non-null amounts.
- `total_amount` and `avg_amount` ignore nulls. A merchant whose amounts are all null gets null, not zero.
- `distinct_methods` counts the distinct non-null methods.
- `ok_payments` and `ok_amount` cover only the payments with status `ok`, using conditional aggregation.

**Sample input**

**`payments`**

| merchant_id | method | amount | status |
| --- | --- | --- | --- |
| m1 | wallet | 10.0 | ok |
| m1 | card | null | failed |
| m1 | card | 30.0 | ok |
| m2 | card | null | failed |
| m2 | card | null | failed |
| m3 | null | 5.0 | ok |
| m3 | wallet | 5.0 | ok |

**Expected output**

| merchant_id | n_rows | n_amount | total_amount | avg_amount | distinct_methods | ok_payments | ok_amount |
| --- | --- | --- | --- | --- | --- | --- | --- |
| m1 | 3 | 2 | 40.0 | 20.0 | 2 | 2 | 40.0 |
| m2 | 2 | 0 | null | null | 1 | 0 | null |
| m3 | 2 | 2 | 10.0 | 5.0 | 1 | 2 | 10.0 |

**Reading the result.** Merchant m2 has two payment rows but no known amounts: its sum and average stay null. Merchant m3 has one distinct non-null method because a null method is not a distinct value.

In [ ]:
# Create the sample DataFrames for this problem.
payments = spark.createDataFrame([
    ("m1", "wallet", 10.0, "ok"),
    ("m1", "card", None, "failed"),
    ("m1", "card", 30.0, "ok"),
    ("m2", "card", None, "failed"),
    ("m2", "card", None, "failed"),
    ("m3", None, 5.0, "ok"),
    ("m3", "wallet", 5.0, "ok"),
], ["merchant_id", "method", "amount", "status"])

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
result = payments.groupBy("merchant_id").agg(
    F.count("*").alias("n_rows"),
    F.count("amount").alias("n_amount"),
    F.sum("amount").alias("total_amount"),
    F.avg("amount").alias("avg_amount"),
    F.countDistinct("method").alias("distinct_methods"),
    F.count(F.when(F.col("status") == "ok", 1)).alias("ok_payments"),
    F.sum(F.when(F.col("status") == "ok", F.col("amount"))).alias("ok_amount"),
).orderBy("merchant_id")

result.show()

## Problem 21 - Subtotals with rollup and grouping

**Difficulty:** Medium · about 15 min

Revenue by region and country. A report needs a line per country, a subtotal per region, and a grand total.

**Statement.** Build the report with one `rollup`, and label each row with its level.

Requirements:

- Use `rollup("region", "country")`, not three separate aggregations.
- Use `grouping()` to tell a real value from a subtotal's null. Do not compare with null.
- Show subtotal and grand total rows with `ALL` in the place of the aggregated column.

**Sample input**

**`revenue`**

| region | country | month | revenue |
| --- | --- | --- | --- |
| EU | DE | 2026-01 | 100.0 |
| EU | DE | 2026-02 | 50.0 |
| EU | FR | 2026-01 | 70.0 |
| US | US | 2026-01 | 200.0 |
| US | US | 2026-02 | 40.0 |

**Expected output**

| region | country | level | revenue |
| --- | --- | --- | --- |
| ALL | ALL | grand_total | 460.0 |
| EU | ALL | region_total | 220.0 |
| EU | DE | country | 150.0 |
| EU | FR | country | 70.0 |
| US | ALL | region_total | 240.0 |
| US | US | country | 240.0 |

**Reading the result.** EU's 220 subtotal combines DE 150 and FR 70. The grand total is 460; ALL labels identify aggregated levels rather than actual country values.

In [ ]:
# Create the sample DataFrames for this problem.
revenue = spark.createDataFrame([
    ("EU", "DE", "2026-01", 100.0),
    ("EU", "DE", "2026-02", 50.0),
    ("EU", "FR", "2026-01", 70.0),
    ("US", "US", "2026-01", 200.0),
    ("US", "US", "2026-02", 40.0),
], ["region", "country", "month", "revenue"])

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
rolled = revenue.rollup("region", "country").agg(
    F.sum("revenue").alias("revenue"),
    F.grouping("region").alias("region_is_total"),
    F.grouping("country").alias("country_is_total"),
)

report = (
    rolled.select(
        F.when(F.col("region_is_total") == 1, "ALL").otherwise(F.col("region")).alias("region"),
        F.when(F.col("country_is_total") == 1, "ALL").otherwise(F.col("country")).alias("country"),
        F.when(F.col("region_is_total") == 1, "grand_total")
        .when(F.col("country_is_total") == 1, "region_total")
        .otherwise("country")
        .alias("level"),
        "revenue",
    )
    .orderBy("region", "country")
)

report.show()

## Problem 22 - Latest customer record with deterministic deduplication

**Difficulty:** Medium · about 15 min

The customer feed contains retries and out-of-order updates. Return exactly one row per `customer_id`, choosing
the latest `updated_at`. If timestamps tie, choose the greatest `ingest_id`. Keep customers whose email is null.

Requirements:

- Do not use `dropDuplicates` without defining which record wins.
- Say how the result changes if two records share `customer_id`, `updated_at` and `ingest_id`. Then the winner is
  arbitrary, so the pipeline should fail a check rather than pick one silently.
- Add a data-quality check that detects ties on the complete winning ordering key before selecting a winner.

**Expected grain:** one row per customer.

**Sample input**

**`customer_updates`**

| customer_id | email | updated_at | ingest_id |
| --- | --- | --- | --- |
| c1 | ava@old.example | 2026-10-01 09:00:00 | 10 |
| c1 | ava@example.com | 2026-10-01 09:00:00 | 11 |
| c2 | null | 2026-10-02 10:00:00 | 20 |
| c2 | ben@example.com | 2026-10-01 10:00:00 | 19 |
| c3 | cy@example.com | 2026-10-03 12:00:00 | 30 |

**Expected output**

| customer_id | email | updated_at | ingest_id |
| --- | --- | --- | --- |
| c1 | ava@example.com | 2026-10-01 09:00:00 | 11 |
| c2 | null | 2026-10-02 10:00:00 | 20 |
| c3 | cy@example.com | 2026-10-03 12:00:00 | 30 |

**Reading the result.** c1's timestamps tie, so ingest ID 11 wins. c2's newest record has a null email and must be kept; a null field is not a reason to fall back to an older version.

In [ ]:
# Create the sample DataFrames for this problem.
customer_updates = spark.createDataFrame([
    ('c1', 'ava@old.example', '2026-10-01 09:00:00', 10),
    ('c1', 'ava@example.com', '2026-10-01 09:00:00', 11),
    ('c2', None, '2026-10-02 10:00:00', 20),
    ('c2', 'ben@example.com', '2026-10-01 10:00:00', 19),
    ('c3', 'cy@example.com', '2026-10-03 12:00:00', 30)
], ['customer_id', 'email', 'updated_at', 'ingest_id'])

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
winning_keys = (customer_updates.withColumn("updated_ts", F.to_timestamp("updated_at"))
    .withColumn("winner_rank", F.dense_rank().over(
        Window.partitionBy("customer_id").orderBy(F.desc("updated_ts"), F.desc("ingest_id"))))
    .filter(F.col("winner_rank") == 1))

latest_customers = (
    customer_updates
    .withColumn("updated_ts", F.to_timestamp("updated_at"))
    .withColumn(
        "rn",
        F.row_number().over(
            Window.partitionBy("customer_id")
            .orderBy(F.col("updated_ts").desc(), F.col("ingest_id").desc())
        ),
    )
    .filter(F.col("rn") == 1)
    .drop("rn", "updated_ts")
)

latest_customers.orderBy("customer_id").show(truncate=False)

## Problem 23 - Spark SQL: CTEs and conditional aggregation

**Difficulty:** Medium · about 15 min

Register the orders as a temporary view. Use a CTE to select paid orders, then return total paid revenue per region and retain only regions with revenue at least 100. The view is session-scoped; no persistent table is required.

**Sample input**

**`sales`**

| region | status | amount |
| --- | --- | --- |
| east | paid | 80 |
| east | paid | 40 |
| west | paid | 90 |
| west | cancelled | 30 |

**Expected output**

| region | revenue |
| --- | --- |
| east | 120 |

**Reading the result.** East has paid revenue 80 + 40 = 120. West's cancelled 30 is excluded, leaving paid revenue 90, below the report's threshold of 100.

In [ ]:
# Create the sample DataFrames for this problem.
sales = spark.createDataFrame([('east', 'paid', 80), ('east', 'paid', 40), ('west', 'paid', 90), ('west', 'cancelled', 30)], ['region', 'status', 'amount'])

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
sales.createOrReplaceTempView('challenge_sales')

result = spark.sql("""WITH paid AS (SELECT region, amount FROM challenge_sales WHERE status = 'paid')
SELECT region, SUM(amount) AS revenue FROM paid GROUP BY region HAVING SUM(amount) >= 100""")

result.show()

## Problem 24 - Window functions: top two per department

**Difficulty:** Medium · about 15 min

Return the top two employees per department by salary. Break ties by employee ID ascending so that exactly two rows are selected where available. Return department, employee_id, salary, and rank.

**Sample input**

**`employees`**

| department | employee_id | salary |
| --- | --- | --- |
| A | 1 | 100 |
| A | 2 | 100 |
| A | 3 | 80 |
| B | 4 | 90 |

**Expected output**

| department | employee_id | salary | rank |
| --- | --- | --- | --- |
| A | 1 | 100 | 1 |
| A | 2 | 100 | 2 |
| B | 4 | 90 | 1 |

**Reading the result.** Employees 1 and 2 tie on salary, so ID ascending determines their positions. Department B contains only one employee and returns one row.

In [ ]:
# Create the sample DataFrames for this problem.
employees = spark.createDataFrame([('A', 1, 100), ('A', 2, 100), ('A', 3, 80), ('B', 4, 90)], ['department', 'employee_id', 'salary'])

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
w = Window.partitionBy('department').orderBy(F.desc('salary'), F.asc('employee_id'))

result = employees.withColumn('rank', F.row_number().over(w)).filter(F.col('rank') <= 2)

result.orderBy('department', 'rank').show()

## Problem 25 - Running totals with an explicit window frame

**Difficulty:** Medium · about 15 min

Compute a running amount per account ordered by transaction date and ID. Treat null amounts as zero. Use a ROWS frame so each transaction advances the total individually.

**Sample input**

**`transactions`**

| account | tx_id | tx_date | amount |
| --- | --- | --- | --- |
| a | 1 | 2026-01-01 | 5 |
| a | 2 | 2026-01-01 | null |
| a | 3 | 2026-01-02 | 7 |
| b | 4 | 2026-01-01 | 2 |

**Expected output**

| account | tx_id | running_total |
| --- | --- | --- |
| a | 1 | 5 |
| a | 2 | 5 |
| a | 3 | 12 |
| b | 4 | 2 |

**Reading the result.** The null amount on transaction 2 adds zero. Because date and transaction ID define the sequence, the two same-day rows have totals 5 and 5 before the next day raises it to 12.

In [ ]:
# Create the sample DataFrames for this problem.
transactions = spark.createDataFrame([('a', 1, '2026-01-01', 5), ('a', 2, '2026-01-01', None), ('a', 3, '2026-01-02', 7), ('b', 4, '2026-01-01', 2)], 'account string, tx_id int, tx_date string, amount int')

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
w = Window.partitionBy('account').orderBy('tx_date', 'tx_id').rowsBetween(Window.unboundedPreceding, Window.currentRow)

result = transactions.select('account', 'tx_id', F.sum(F.coalesce('amount', F.lit(0))).over(w).alias('running_total'))

result.orderBy('account', 'tx_id').show()

## Problem 26 - Union evolving schemas by name

**Difficulty:** Medium · about 15 min

Combine the two feeds by column name. The old feed has no `source` column: represent that value as null. Return `id`, `amount`, `source`.

**Sample input**

**`old_feed`**

| id | amount |
| --- | --- |
| 1 | 10 |

**`new_feed`**

| source | id | amount |
| --- | --- | --- |
| web | 2 | 20 |

**Expected output**

| id | amount | source |
| --- | --- | --- |
| 1 | 10 | null |
| 2 | 20 | web |

**Reading the result.** The old row has no source field, so its source is null. The new feed lists its columns in a different order; matching by name keeps the ID and amount in the right columns.

In [ ]:
# Create the sample DataFrames for this problem.
old_feed = spark.createDataFrame([(1, 10)], ['id', 'amount'])
new_feed = spark.createDataFrame([('web', 2, 20)], ['source', 'id', 'amount'])

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
result = old_feed.unionByName(new_feed, allowMissingColumns=True).select('id', 'amount', 'source')

result.orderBy('id').show()

## Problem 27 - Fill missing days in a daily series

**Difficulty:** Medium · about 15 min

A dashboard plots daily revenue per store, and days without sales must show as zero instead of disappearing.

**Statement.** Return one row per store and day, from the store's first to its last sale date, with `store`,
`sale_date`, `revenue` and `is_filled`, ordered by `store` and `sale_date`.

Requirements:

- Each store gets its own date range; do not invent days before its first or after its last sale.
- A missing day has `revenue` 0.0 and `is_filled` true; a real day keeps its revenue and has `is_filled` false.
- Build the calendar with `sequence` and `explode`, not with a Python loop.

**Expected grain:** one row per store and calendar day.

**Sample input**

**`daily_sales`**

| store | sale_date | revenue |
| --- | --- | --- |
| s1 | 2026-01-01 | 100.0 |
| s1 | 2026-01-03 | 50.0 |
| s2 | 2026-01-02 | 70.0 |

**Expected output**

| store | sale_date | revenue | is_filled |
| --- | --- | --- | --- |
| s1 | 2026-01-01 | 100.0 | false |
| s1 | 2026-01-02 | 0.0 | true |
| s1 | 2026-01-03 | 50.0 | false |
| s2 | 2026-01-02 | 70.0 | false |

**Reading the result.** Store `s1` had no sale on 2 January, so that day is filled with zero. Store `s2` sold on one
day only, so its range is a single day.

In [ ]:
# Create the sample DataFrames for this problem.
daily_sales = spark.createDataFrame([
    ("s1", "2026-01-01", 100.0),
    ("s1", "2026-01-03", 50.0),
    ("s2", "2026-01-02", 70.0),
], "store string, sale_date string, revenue double")

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
sales = daily_sales.withColumn("sale_date", F.to_date("sale_date"))

bounds = sales.groupBy("store").agg(F.min("sale_date").alias("first_day"), F.max("sale_date").alias("last_day"))

calendar = bounds.select("store", F.explode(F.sequence("first_day", "last_day")).alias("sale_date"))

result = (
    calendar.join(sales, ["store", "sale_date"], "left")
    .select(
        "store",
        "sale_date",
        F.coalesce("revenue", F.lit(0.0)).alias("revenue"),
        F.col("revenue").isNull().alias("is_filled"),
    )
    .orderBy("store", "sale_date")
)

result.show()

## Problem 28 - Median and percentiles per group

**Difficulty:** Medium · about 15 min

Operations reports delivery times per region. Averages hide long tails, so the report uses the median and the
90th percentile.

**Statement.** Return `region`, `deliveries`, `median_minutes` and `p90_minutes`, ordered by `region`.

Requirements:

- `median_minutes` is the exact median; with an even count it is the mean of the two middle values.
- `p90_minutes` uses `percentile_approx`, which returns an actual value from the data.
- Be ready to explain when you would accept an approximate percentile on a large table, and why.

**Expected grain:** one row per region.

**Sample input**

**`deliveries`**

| region | minutes |
| --- | --- |
| north | 10 |
| north | 20 |
| north | 30 |
| north | 40 |
| south | 5 |
| south | 15 |
| south | 100 |

**Expected output**

| region | deliveries | median_minutes | p90_minutes |
| --- | --- | --- | --- |
| north | 4 | 25.0 | 40 |
| south | 3 | 15.0 | 100 |

**Reading the result.** North has four deliveries, so its median is the mean of 20 and 30. South's single slow
delivery barely moves the median but sets the 90th percentile, which is exactly what the average would hide.

In [ ]:
# Create the sample DataFrames for this problem.
deliveries = spark.createDataFrame([
    ("north", 10), ("north", 20), ("north", 30), ("north", 40),
    ("south", 5), ("south", 15), ("south", 100),
], "region string, minutes int")

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
result = (
    deliveries.groupBy("region")
    .agg(
        F.count("*").alias("deliveries"),
        F.median("minutes").alias("median_minutes"),
        F.percentile_approx("minutes", 0.9).alias("p90_minutes"),
    )
    .orderBy("region")
)

result.show()

## Problem 29 - Parse JSON payloads and summarize arrays without exploding

**Difficulty:** Medium · about 20 min

A payment provider posts webhooks as JSON strings. Some have no `items`, some have an empty list, and one is not
JSON at all.

**Statement.** Return `event_id`, `customer`, `item_count`, `units`, `order_value` and `is_valid`, ordered by
`event_id`.

Requirements:

- Parse `payload` with `from_json` and an explicit schema.
- Compute `item_count`, `units` (sum of `qty`) and `order_value` (sum of `qty * price`) with higher-order
  functions such as `aggregate`, without `explode`, so every event keeps exactly one row.
- A missing or empty `items` list counts as zero items, zero units and 0.0 value.
- `is_valid` is false when the payload could not be parsed; its `customer` is null.

**Expected grain:** one row per webhook event.

**Sample input**

**`webhooks`**

| event_id | payload |
| --- | --- |
| w1 | `{"customer": "c1", "items": [{"sku": "A", "qty": 2, "price": 5.0}, {"sku": "B", "qty": 1, "price": 12.5}]}` |
| w2 | `{"customer": "c2", "items": []}` |
| w3 | `{"customer": "c3"}` |
| w4 | `not json` |

**Expected output**

| event_id | customer | item_count | units | order_value | is_valid |
| --- | --- | --- | --- | --- | --- |
| w1 | c1 | 2 | 3 | 22.5 | true |
| w2 | c2 | 0 | 0 | 0.0 | true |
| w3 | c3 | 0 | 0 | 0.0 | true |
| w4 | null | 0 | 0 | 0.0 | false |

**Reading the result.** `w1` has two lines worth 2 x 5.0 + 1 x 12.5 = 22.5. `w2` and `w3` stay in the output with
zeros; `explode` would have dropped both. `w4` is kept and flagged so it can be quarantined.

In [ ]:
# Create the sample DataFrames for this problem.
webhooks = spark.createDataFrame([
    ("w1", '{"customer": "c1", "items": [{"sku": "A", "qty": 2, "price": 5.0}, {"sku": "B", "qty": 1, "price": 12.5}]}'),
    ("w2", '{"customer": "c2", "items": []}'),
    ("w3", '{"customer": "c3"}'),
    ("w4", "not json"),
], "event_id string, payload string")

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
payload_schema = "customer string, items array<struct<sku:string, qty:int, price:double>>"

parsed = webhooks.withColumn("data", F.from_json("payload", payload_schema))

items = F.col("data.items")

result = (
    parsed.select(
        "event_id",
        F.col("data.customer").alias("customer"),
        F.when(items.isNull(), F.lit(0)).otherwise(F.size(items)).alias("item_count"),
        F.coalesce(F.aggregate(items, F.lit(0), lambda total, i: total + i["qty"]), F.lit(0)).alias("units"),
        F.coalesce(
            F.aggregate(items, F.lit(0.0), lambda total, i: total + i["qty"] * i["price"]), F.lit(0.0)
        ).alias("order_value"),
        F.col("data.customer").isNotNull().alias("is_valid"),
    )
    .orderBy("event_id")
)

result.show(truncate=False)

## Problem 30 - Non-equi joins: price bands and promotions valid on a date

**Difficulty:** Hard · about 25 min

Orders need two lookups that are not equality joins. A price band is a range of amounts. A promotion is valid for a
range of dates.

**Statement.** Give each order its price band and its promotion code, if any.

Requirements:

- A band holds amounts from `min_amount` inclusive up to `max_amount` exclusive. The top band has a null `max_amount`
  and no upper limit.
- A promotion holds for `valid_from` through `valid_to`, both inclusive.
- Keep every order, even one with no promotion. Each order must appear exactly once.

**Sample input**

**`orders_nq`**

| order_id | amount | order_date |
| --- | --- | --- |
| 1 | 30.0 | 2026-05-02 |
| 2 | 50.0 | 2026-05-10 |
| 3 | 250.0 | 2026-06-01 |
| 4 | 120.0 | 2026-06-20 |
| 5 | 10.0 | 2026-07-04 |

**`bands_nq`**

| band | min_amount | max_amount |
| --- | --- | --- |
| basic | 0.0 | 50.0 |
| plus | 50.0 | 200.0 |
| pro | 200.0 | null |

**`promos_nq`**

| promo_code | valid_from | valid_to |
| --- | --- | --- |
| SPRING10 | 2026-05-01 | 2026-05-31 |
| SUMMER15 | 2026-06-01 | 2026-06-30 |

**Expected output**

| order_id | amount | band | promo_code |
| --- | --- | --- | --- |
| 1 | 30.0 | basic | SPRING10 |
| 2 | 50.0 | plus | SPRING10 |
| 3 | 250.0 | pro | SUMMER15 |
| 4 | 120.0 | plus | SUMMER15 |
| 5 | 10.0 | basic | null |

**Reading the result.** The amount 50 belongs to plus because a band's lower bound is inclusive and upper bound exclusive. July 4 has no promotion, but its order still appears.

In [ ]:
# Create the sample DataFrames for this problem.
orders_nq = spark.createDataFrame([
    (1, 30.0, "2026-05-02"),
    (2, 50.0, "2026-05-10"),
    (3, 250.0, "2026-06-01"),
    (4, 120.0, "2026-06-20"),
    (5, 10.0, "2026-07-04"),
], ["order_id", "amount", "order_date"])
bands_nq = spark.createDataFrame([
    ("basic", 0.0, 50.0),
    ("plus", 50.0, 200.0),
    ("pro", 200.0, None),
], ["band", "min_amount", "max_amount"])
promos_nq = spark.createDataFrame([
    ("SPRING10", "2026-05-01", "2026-05-31"),
    ("SUMMER15", "2026-06-01", "2026-06-30"),
], ["promo_code", "valid_from", "valid_to"])

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
orders_dated = orders_nq.withColumn("order_date", F.to_date("order_date"))

promos_dated = (
    promos_nq
    .withColumn("valid_from", F.to_date("valid_from"))
    .withColumn("valid_to", F.to_date("valid_to"))
)

banded = (
    orders_dated.alias("o")
    .join(
        bands_nq.alias("b"),
        (F.col("o.amount") >= F.col("b.min_amount"))
        & (F.col("b.max_amount").isNull() | (F.col("o.amount") < F.col("b.max_amount"))),
        "inner",
    )
    .select(
        F.col("o.order_id").alias("order_id"),
        F.col("o.amount").alias("amount"),
        F.col("o.order_date").alias("order_date"),
        F.col("b.band").alias("band"),
    )
)

priced = (
    banded.alias("x")
    .join(
        promos_dated.alias("p"),
        (F.col("x.order_date") >= F.col("p.valid_from")) & (F.col("x.order_date") <= F.col("p.valid_to")),
        "left",
    )
    .select("x.order_id", "x.amount", "x.band", "p.promo_code")
    .orderBy("order_id")
)

priced.show()

## Problem 31 - first_value, last_value frames and change against the first value

**Difficulty:** Hard · about 25 min

Each SKU has a series of price changes. A naive `last_value` returns the current row, which is a common mistake.

**Statement.** For each price change, return the first and the last price of its SKU, and the change in percent from
the first price.

Requirements:

- Give `last_value` a frame that covers the whole partition. Without it the default frame ends at the current row.
- `pct_vs_first` is `(price - first_price) / first_price * 100`, rounded to two decimals.
- Show that a naive `last` over the ordered window returns each row's own price.

**Sample input**

**`price_changes`**

| sku | effective_on | price |
| --- | --- | --- |
| sku1 | 2026-01-01 | 10.0 |
| sku1 | 2026-02-01 | 12.0 |
| sku1 | 2026-03-01 | 9.0 |
| sku2 | 2026-01-15 | 5.0 |
| sku2 | 2026-04-01 | 6.0 |

**Expected output**

| sku | effective_on | price | first_price | last_price | pct_vs_first |
| --- | --- | --- | --- | --- | --- |
| sku1 | 2026-01-01 | 10.0 | 10.0 | 9.0 | 0.0 |
| sku1 | 2026-02-01 | 12.0 | 10.0 | 9.0 | 20.0 |
| sku1 | 2026-03-01 | 9.0 | 10.0 | 9.0 | -10.0 |
| sku2 | 2026-01-15 | 5.0 | 5.0 | 6.0 | 0.0 |
| sku2 | 2026-04-01 | 6.0 | 5.0 | 6.0 | 20.0 |

**Reading the result.** Every sku1 row must show a final price of 9, including January's row. The percentage compares each current price with the first price of 10.

In [ ]:
# Create the sample DataFrames for this problem.
price_changes = spark.createDataFrame([
    ("sku1", "2026-01-01", 10.0),
    ("sku1", "2026-02-01", 12.0),
    ("sku1", "2026-03-01", 9.0),
    ("sku2", "2026-01-15", 5.0),
    ("sku2", "2026-04-01", 6.0),
], ["sku", "effective_on", "price"])

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
ordered = Window.partitionBy("sku").orderBy("effective_on")

whole_sku = ordered.rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)

dated = price_changes.withColumn("effective_on", F.to_date("effective_on"))

naive_last = dated.withColumn("naive_last", F.last("price").over(ordered))

result = (
    dated
    .withColumn("first_price", F.first("price").over(whole_sku))
    .withColumn("last_price", F.last("price").over(whole_sku))
    .withColumn("pct_vs_first", F.round((F.col("price") - F.col("first_price")) / F.col("first_price") * 100, 2))
    .select("sku", "effective_on", "price", "first_price", "last_price", "pct_vs_first")
    .orderBy("sku", "effective_on")
)

result.show()

naive_last.orderBy("sku", "effective_on").show()

## Problem 32 - Orders joined to a many-to-many category dimension

**Difficulty:** Hard · about 25 min

Produce one row per order with `order_total`, `discount_total` and `net_total`. Each order has line items, and a
product can carry several category labels. The labels feed a separate category summary, but they must not
multiply order revenue.

Requirements:

- Compute line revenue at the line grain before any join to categories.
- Keep orders whose product has no category.
- Build the category summary separately, one row per category. A line in two categories counts in both, so
  category totals are not additive. Say so in the answer.
- Prove that adding the category relationship leaves the sum of order totals unchanged.

**Expected grains:** one row per order for the order summary; one row per category for the category summary.

**Sample input**

**`order_lines`**

| order_id | product_id | quantity | unit_price |
| --- | --- | --- | --- |
| o1 | p1 | 2 | 10.0 |
| o1 | p2 | 1 | 25.0 |
| o2 | p1 | 1 | 10.0 |
| o3 | p3 | 1 | 8.0 |

**`discounts`**

| order_id | discount_id | discount_amount |
| --- | --- | --- |
| o1 | d1 | 5.0 |
| o1 | d2 | 2.0 |
| o3 | d3 | 1.0 |

**`product_categories`**

| product_id | category |
| --- | --- |
| p1 | home |
| p1 | sale |
| p2 | home |

**Expected output**

**Order summary**

| order_id | order_total | discount_total | net_total |
| --- | --- | --- | --- |
| o1 | 45.0 | 7.0 | 38.0 |
| o2 | 10.0 | 0.0 | 10.0 |
| o3 | 8.0 | 1.0 | 7.0 |

**Category summary**

| category | category_revenue |
| --- | --- |
| home | 55.0 |
| sale | 30.0 |

**Reading the result.** o1 has line revenue 45 and total discounts 7, giving net revenue 38. Product p1 belongs to two categories, so its revenue appears in both category totals but only once in the order report.

In [ ]:
# Create the sample DataFrames for this problem.
order_lines = spark.createDataFrame([
    ('o1', 'p1', 2, 10.0), ('o1', 'p2', 1, 25.0),
    ('o2', 'p1', 1, 10.0), ('o3', 'p3', 1, 8.0)
], ['order_id', 'product_id', 'quantity', 'unit_price'])
discounts = spark.createDataFrame([
    ('o1', 'd1', 5.0), ('o1', 'd2', 2.0), ('o3', 'd3', 1.0)
], ['order_id', 'discount_id', 'discount_amount'])
product_categories = spark.createDataFrame([
    ('p1', 'home'), ('p1', 'sale'), ('p2', 'home')
], ['product_id', 'category'])

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
lines = order_lines.withColumn("line_revenue", F.col("quantity") * F.col("unit_price"))

discount_by_order = discounts.groupBy("order_id").agg(F.sum("discount_amount").alias("discount_total"))

order_summary = (
    lines.groupBy("order_id").agg(F.sum("line_revenue").alias("order_total"))
    .join(discount_by_order, "order_id", "left")
    .withColumn("discount_total", F.coalesce("discount_total", F.lit(0.0)))
    .withColumn("net_total", F.col("order_total") - F.col("discount_total"))
    .orderBy("order_id")
)

order_summary.show()

category_summary = (
    lines.join(product_categories, "product_id", "inner")
    .groupBy("category").agg(F.sum("line_revenue").alias("category_revenue"))
    .orderBy("category")
)

category_summary.show()

order_revenue = order_summary.agg(F.sum("order_total")).first()[0]

line_revenue = lines.agg(F.sum("line_revenue")).first()[0]

print("Order revenue:", order_revenue, "Line revenue:", line_revenue)

## Problem 33 - Slowly changing dimension as-of join

**Difficulty:** Hard · about 25 min

Attach the customer segment that was valid when each event occurred. Dimension versions use the half-open interval
`[effective_from, effective_to)`, and a null `effective_to` means the version is current.

Requirements:

- Keep events that have no matching version.
- An event exactly at `effective_to` belongs to the next version.
- Detect overlapping versions of the same customer before the join, because overlaps would fan out the result.
- Return exactly one row per event.

**Expected grain:** one row per event.

**Sample input**

**`events`**

| event_id | customer_id | event_ts | amount |
| --- | --- | --- | --- |
| e1 | c1 | 2026-01-15 12:00:00 | 20.0 |
| e2 | c1 | 2026-07-01 12:00:00 | 30.0 |
| e3 | c2 | 2026-03-01 12:00:00 | 40.0 |
| e4 | missing | 2026-03-01 12:00:00 | 50.0 |

**`segment_history`**

| customer_id | segment | effective_from | effective_to |
| --- | --- | --- | --- |
| c1 | standard | 2026-01-01 00:00:00 | 2026-06-01 00:00:00 |
| c1 | premium | 2026-06-01 00:00:00 | null |
| c2 | standard | 2026-01-01 00:00:00 | null |

**Expected output**

| event_id | customer_id | event_ts | amount | segment |
| --- | --- | --- | --- | --- |
| e1 | c1 | 2026-01-15 12:00:00 | 20.0 | standard |
| e2 | c1 | 2026-07-01 12:00:00 | 30.0 | premium |
| e3 | c2 | 2026-03-01 12:00:00 | 40.0 | standard |
| e4 | missing | 2026-03-01 12:00:00 | 50.0 | null |

**Reading the result.** e1 uses c1's standard version; e2 uses premium. The unmatched customer stays in the output with a null segment. The exclusive end boundary prevents adjacent versions from both matching.

In [ ]:
# Create the sample DataFrames for this problem.
events = spark.createDataFrame([
    ('e1', 'c1', '2026-01-15 12:00:00', 20.0),
    ('e2', 'c1', '2026-07-01 12:00:00', 30.0),
    ('e3', 'c2', '2026-03-01 12:00:00', 40.0),
    ('e4', 'missing', '2026-03-01 12:00:00', 50.0)
], ['event_id', 'customer_id', 'event_ts', 'amount'])
segment_history = spark.createDataFrame([
    ('c1', 'standard', '2026-01-01 00:00:00', '2026-06-01 00:00:00'),
    ('c1', 'premium', '2026-06-01 00:00:00', None),
    ('c2', 'standard', '2026-01-01 00:00:00', None)
], ['customer_id', 'segment', 'effective_from', 'effective_to'])

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
if not (segment_history.groupBy('customer_id', 'effective_from').count().filter(F.col('count') > 1).count() == 0):
    raise ValueError("Ambiguous or overlapping source versions")

history = (segment_history
    .withColumn("effective_from", F.to_timestamp("effective_from"))
    .withColumn("effective_to", F.to_timestamp("effective_to")))

events_ts = events.withColumn("event_ts", F.to_timestamp("event_ts"))

a = history.alias("a")

b = history.alias("b")

overlapping = a.join(
    b,
    (F.col("a.customer_id") == F.col("b.customer_id"))
    & (F.col("a.effective_from") < F.col("b.effective_from"))
    & (F.col("b.effective_from") < F.coalesce(F.col("a.effective_to"), MAX_TS)),
)

if not (overlapping.count() == 0):
    raise ValueError("Ambiguous or overlapping source versions")

in_version = (
    (F.col("e.customer_id") == F.col("h.customer_id"))
    & (F.col("e.event_ts") >= F.col("h.effective_from"))
    & (F.col("e.event_ts") < F.coalesce(F.col("h.effective_to"), MAX_TS))
)

priced = (events_ts.alias("e")
    .join(history.alias("h"), in_version, "left")
    .select("e.event_id", "e.customer_id", "e.event_ts", "e.amount", "h.segment"))

priced.orderBy("event_id").show(truncate=False)

segments = {r["event_id"]: r["segment"] for r in priced.collect()}

boundary = (spark.createDataFrame([("eb", "c1", "2026-06-01 00:00:00", 1.0)],
                                  ["event_id", "customer_id", "event_ts", "amount"])
    .withColumn("event_ts", F.to_timestamp("event_ts")))

boundary_row = (boundary.alias("e")
    .join(history.alias("h"), in_version, "left")
    .select("h.segment").first())
print("segments:", segments)
print("an event exactly at effective_to gets:", boundary_row["segment"])

## Problem 34 - Sessionize events without leaking across users

**Difficulty:** Hard · about 25 min

Create sessions from web events. A new session starts when the gap from the previous event of the same user is
greater than 30 minutes (exactly 30 minutes stays in the session). Return one row per session with its start,
end, event count and total value.

Requirements:

- Window by user and order by timestamp, then event id, so the result is deterministic.
- Events of one user must never affect another user's sessions.
- A null value counts as zero in the session total.
- Explain the cost of the window and how you would scale it when one user has billions of events.

**Expected grain:** one row per user session.

**Sample input**

**`web_events`**

| user_id | event_id | event_ts | value |
| --- | --- | --- | --- |
| u1 | a | 2026-10-01 10:00:00 | 4.0 |
| u1 | b | 2026-10-01 10:20:00 | null |
| u1 | c | 2026-10-01 11:00:01 | 7.0 |
| u2 | d | 2026-10-01 10:05:00 | 3.0 |
| u2 | e | 2026-10-01 10:35:00 | 5.0 |

**Expected output**

| user_id | session_no | session_start | session_end | event_count | session_value |
| --- | --- | --- | --- | --- | --- |
| u1 | 1 | 2026-10-01 10:00:00 | 2026-10-01 10:20:00 | 2 | 4.0 |
| u1 | 2 | 2026-10-01 11:00:01 | 2026-10-01 11:00:01 | 1 | 7.0 |
| u2 | 1 | 2026-10-01 10:05:00 | 2026-10-01 10:35:00 | 2 | 8.0 |

**Reading the result.** u1's gap before 11:00:01 exceeds 30 minutes and starts session 2. u2's exactly 30-minute gap stays in one session. Null event value contributes zero without removing its event.

In [ ]:
# Create the sample DataFrames for this problem.
web_events = spark.createDataFrame([
    ('u1', 'a', '2026-10-01 10:00:00', 4.0),
    ('u1', 'b', '2026-10-01 10:20:00', None),
    ('u1', 'c', '2026-10-01 11:00:01', 7.0),
    ('u2', 'd', '2026-10-01 10:05:00', 3.0),
    ('u2', 'e', '2026-10-01 10:35:00', 5.0)
], ['user_id', 'event_id', 'event_ts', 'value'])

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
ordered = Window.partitionBy("user_id").orderBy("event_ts", "event_id")

events_ts = web_events.withColumn("event_ts", F.to_timestamp("event_ts"))

with_gap = events_ts.withColumn("prev_ts", F.lag("event_ts").over(ordered))

is_new_session = F.when(
    F.col("prev_ts").isNull()
    | (F.unix_timestamp("event_ts") - F.unix_timestamp("prev_ts") > 30 * 60),
    1,
).otherwise(0)

sessions = (
    with_gap.withColumn("is_new", is_new_session)
    .withColumn("session_no", F.sum("is_new").over(ordered.rowsBetween(Window.unboundedPreceding, Window.currentRow)))
    .groupBy("user_id", "session_no")
    .agg(
        F.min("event_ts").alias("session_start"),
        F.max("event_ts").alias("session_end"),
        F.count("*").alias("event_count"),
        F.sum(F.coalesce("value", F.lit(0.0))).alias("session_value"),
    )
    .orderBy("user_id", "session_no")
)

sessions.show(truncate=False)

## Problem 35 - Skewed join with selective salting

**Difficulty:** Hard · about 25 min

A fact table is joined to a dimension on `account_id`. Profiling shows that `UNKNOWN` holds most fact rows, so one
task would do most of the join work. Implement a salted join that gives the same result as the unsalted join.

Requirements:

- Salt only the hot key, with a deterministic bucket from `event_id`.
- Replicate only the hot dimension row across the salt range, not the whole dimension.
- Compare row counts and totals with a baseline unsalted inner join.
- Say when broadcasting the dimension or enabling AQE skew join would be better.
- Say how to choose the salt count from Spark UI metrics (task durations and shuffle read sizes), not by guessing.

**Expected grain:** one row per matching fact row.

**Sample input**

**`fact_events`**

| event_id | account_id | metric |
| --- | --- | --- |
| e1 | UNKNOWN | 10 |
| e2 | UNKNOWN | 20 |
| e3 | UNKNOWN | 30 |
| e4 | a1 | 5 |
| e5 | a2 | 7 |
| e6 | null | 11 |

**`accounts`**

| account_id | account_type |
| --- | --- |
| UNKNOWN | fallback |
| a1 | alpha |
| a2 | beta |

**Expected output**

| event_id | account_id | metric | account_type |
| --- | --- | --- | --- |
| e1 | UNKNOWN | 10 | fallback |
| e2 | UNKNOWN | 20 | fallback |
| e3 | UNKNOWN | 30 | fallback |
| e4 | a1 | 5 | alpha |
| e5 | a2 | 7 | beta |

**Join totals**

| join | rows | metric_total |
| --- | --- | --- |
| baseline | 5 | 72 |
| salted | 5 | 72 |

**Reading the result.** The salted join keeps the same five matched events and metric total 72. The null account event is absent from both inner joins. Salting changes work distribution, not the business result.

In [ ]:
# Create the sample DataFrames for this problem.
fact_events = spark.createDataFrame([
    ('e1', 'UNKNOWN', 10), ('e2', 'UNKNOWN', 20), ('e3', 'UNKNOWN', 30),
    ('e4', 'a1', 5), ('e5', 'a2', 7), ('e6', None, 11)
], ['event_id', 'account_id', 'metric'])
accounts = spark.createDataFrame([
    ('UNKNOWN', 'fallback'), ('a1', 'alpha'), ('a2', 'beta')
], ['account_id', 'account_type'])

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
SALT_BUCKETS = 3

HOT_KEY = "UNKNOWN"

baseline = fact_events.join(accounts, "account_id", "inner")

fact_salted = fact_events.withColumn(
    "salt",
    F.when(F.col("account_id") == HOT_KEY, F.pmod(F.hash("event_id"), F.lit(SALT_BUCKETS)))
    .otherwise(F.lit(0)),
)

salts = spark.range(SALT_BUCKETS).select(F.col("id").cast("int").alias("salt"))

hot_dimension = accounts.filter(F.col("account_id") == HOT_KEY).crossJoin(salts)

cold_dimension = accounts.filter(F.col("account_id") != HOT_KEY).withColumn("salt", F.lit(0))

dimension_salted = hot_dimension.unionByName(cold_dimension)

salted = (
    fact_salted.join(dimension_salted, ["account_id", "salt"], "inner")
    .select("event_id", "account_id", "metric", "account_type")
)

salted.orderBy("event_id").show()

join_totals = (
    baseline.agg(F.count("*").alias("rows"), F.sum("metric").alias("metric_total"))
    .withColumn("join", F.lit("baseline"))
    .unionByName(salted.agg(F.count("*").alias("rows"), F.sum("metric").alias("metric_total"))
                 .withColumn("join", F.lit("salted")))
    .select("join", "rows", "metric_total").orderBy("join")
)
join_totals.show()

## Problem 36 - Longest streak of consecutive active days

**Difficulty:** Hard · about 25 min

Product wants each user's longest run of consecutive days with at least one login: a classic gaps-and-islands
problem.

**Statement.** Return `user_id`, `longest_streak`, `streak_start` and `streak_end`, ordered by `user_id`.

Requirements:

- Several logins on the same day count as one active day.
- A streak is a run of calendar days with no gap; a missing day ends it.
- When a user has two streaks of the same length, return the earlier one.
- Use window functions; do not collect rows to Python.

**Expected grain:** one row per user.

**Sample input**

**`user_logins`**

| user_id | login_date |
| --- | --- |
| u1 | 2026-01-01 |
| u1 | 2026-01-02 |
| u1 | 2026-01-02 |
| u1 | 2026-01-03 |
| u1 | 2026-01-05 |
| u1 | 2026-01-06 |
| u2 | 2026-01-10 |
| u3 | 2026-01-01 |
| u3 | 2026-01-03 |

**Expected output**

| user_id | longest_streak | streak_start | streak_end |
| --- | --- | --- | --- |
| u1 | 3 | 2026-01-01 | 2026-01-03 |
| u2 | 1 | 2026-01-10 | 2026-01-10 |
| u3 | 1 | 2026-01-01 | 2026-01-01 |

**Reading the result.** Subtracting each day's row number from the date gives the same value for every day in a
run, so grouping by that value finds the islands. `u1`'s duplicate login on 2 January counts once; `u3` has two
one-day streaks and the earlier one wins.

In [ ]:
# Create the sample DataFrames for this problem.
user_logins = spark.createDataFrame([
    ("u1", "2026-01-01"), ("u1", "2026-01-02"), ("u1", "2026-01-02"), ("u1", "2026-01-03"),
    ("u1", "2026-01-05"), ("u1", "2026-01-06"), ("u2", "2026-01-10"),
    ("u3", "2026-01-01"), ("u3", "2026-01-03"),
], "user_id string, login_date string")

In [ ]:
# Write your DataFrame or Spark SQL solution here.

In [ ]:
days = user_logins.select("user_id", F.to_date("login_date").alias("day")).distinct()

by_day = Window.partitionBy("user_id").orderBy("day")

islands = days.withColumn("island", F.date_sub("day", F.row_number().over(by_day)))

streaks = islands.groupBy("user_id", "island").agg(
    F.count("*").alias("streak"), F.min("day").alias("streak_start"), F.max("day").alias("streak_end")
)

longest_first = Window.partitionBy("user_id").orderBy(F.desc("streak"), "streak_start")

result = (
    streaks.withColumn("rn", F.row_number().over(longest_first))
    .where("rn = 1")
    .select("user_id", F.col("streak").alias("longest_streak"), "streak_start", "streak_end")
    .orderBy("user_id")
)

result.show()